# ESM PLL Trial

Quick verification notebook for ESM2 pseudo log-likelihood (PLL) scoring.

This notebook uses the lightest ESM2 model (`esm2_t6_8M_UR50D`) and checks the filtering rule:
`reject if PLL < -0.5`.

In [1]:
import os
import sys
from pathlib import Path

In [2]:
# Make src importable when running from analysis/notebooks/dev
project_root = Path.cwd().resolve()
if project_root.name == "dev":
    project_root = project_root.parents[2]
elif project_root.name == "notebooks":
    project_root = project_root.parents[1]

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Src path added: {src_path}")

Project root: C:\Users\Karol\Desktop\PepCompass\pep-compass
Src path added: C:\Users\Karol\Desktop\PepCompass\pep-compass\src


In [3]:
# Install once per environment if needed:
# %pip install fair-esm

import torch
from pep_compass.models.esm import ESM2PPLScorer

In [4]:
ESM_MODEL = "esm2_t6_8M_UR50D"
THRESHOLD = -0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
scorer = ESM2PPLScorer(model_name=ESM_MODEL, device=DEVICE)
print(f"Loaded model: {ESM_MODEL}")

Using device: cpu
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to C:\Users\Karol/.cache\torch\hub\checkpoints\esm2_t6_8M_UR50D.pt


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to C:\Users\Karol/.cache\torch\hub\checkpoints\esm2_t6_8M_UR50D-contact-regression.pt


Loaded model: esm2_t6_8M_UR50D


In [5]:
test_sequences = {
    "KY14": "KYCRRFRWLTFRWL",
    "jurand-2": "KFRNRHRWKFKLIFRN",
    "hydrodamin-2": "RMARNLVRYVQGLKKKKVI",
}

for name, seq in test_sequences.items():
    pll = scorer.pll(seq)
    decision = "ACCEPT" if pll >= THRESHOLD else "REJECT"
    print(f"{name:12s} {seq:24s} PLL={pll:8.4f}  => {decision}")

KY14         KYCRRFRWLTFRWL           PLL= -0.6860  => REJECT
jurand-2     KFRNRHRWKFKLIFRN         PLL= -0.6551  => REJECT
hydrodamin-2 RMARNLVRYVQGLKKKKVI      PLL= -0.4804  => ACCEPT


In [6]:
# Single-sequence helper for quick checks
def check_sequence(sequence: str, threshold: float = -0.5):
    pll = scorer.pll(sequence)
    accepted = pll >= threshold
    print({"sequence": sequence, "pll": pll, "threshold": threshold, "accepted": accepted})
    return pll, accepted

# Example:
# check_sequence("KYCRRFRWLTFRWL")